In [ ]:
# Importar dependências

# Gerenciamento de dados e viz
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Progress bar e config
from tqdm.auto import tqdm  # Barra de progresso
from dotenv import load_dotenv  # Carregar env vars

# Métricas de classificação (evaluation)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)

# SDK Typesafe para chamadas ao Jev
from typesafe_sdk import TypeSafeClient, Noul

# Classificação de Tweets com Jev

Avaliação do modelo Jev em tarefa de classificação binária: detectar tweets que relatam desastres reais.

**Dataset**: tweets rotulados (desastre/não-desastre)  
**Abordagem**: Usar Jev com tipo `Noul` (probabilidade) para cada tweet em batch  
**Objetivo**: Medir acurácia, comparar com baseline, otimizar threshold

In [ ]:
# Configurações de predição
BATCH_SIZE = 32      # Processar 32 tweets por chamada à API (eficiência)
THRESHOLD = 0.5      # Threshold inicial para converter probabilidade em classe binária

In [ ]:
# Carregar variáveis de ambiente para acesso à API
load_dotenv()

In [ ]:
# Carregar dataset de tweets com label (desastre: 0/1)
df = pd.read_csv("disaster_tweets.csv")

df.head()

In [ ]:
# Examinar estrutura: 7613 tweets, algumas colunas com missing values
df.info()

In [ ]:
# Distribuição de classes: desbalanceado (57% não-desastre, 43% desastre)
df["target"].value_counts()

In [ ]:
# Proporção (normalizado)
df["target"].value_counts(normalize=True)

In [ ]:
# Visualizar distribuição de classes
sns.countplot(
    data=df,
    x="target"
)

plt.title("Distribuição das classes")
plt.xlabel("Target")
plt.ylabel("Quantidade")
plt.show()

In [ ]:
## Preparação do Dataset

Split 80/20 com estratificação para manter proporções de classe
<cell_type>markdown</cell_type>

# Separar 20% para teste (stratified split)
_, df_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["target"]
)

In [ ]:
# Reset índices após split
df_test = df_test.reset_index(drop=True)

print(df_test.shape)  # 1523 tweets (20% do dataset)
df_test.head()

In [ ]:
# Usar apenas 200 tweets para avaliação (por custo de API)
N_SAMPLES = 200

In [ ]:
# Extrair subset estratificado de N_SAMPLES tweets para avaliação
df_eval, _ = train_test_split(
    df_test,
    train_size=N_SAMPLES,
    random_state=42,
    stratify=df_test["target"]
)

df_eval = df_eval.reset_index(drop=True)

In [ ]:
## Funções de Predição

def predict_batch(client, batch):
    """
    Enviar batch de tweets para Jev para classificação.
    
    Estratégia: Uma pergunta Noul por tweet (P(desastre)).
    Jev responde tudo em uma única chamada.
    """
    
    # Estruturar input: lista de tweets com metadados
    state = {
        "tweets": [
            {
                "text": row["text"],
                "keyword": (
                    row["keyword"]
                    if pd.notna(row["keyword"])
                    else None
                ),
                "location": (
                    row["location"]
                    if pd.notna(row["location"])
                    else None
                ),
            }
            for _, row in batch.iterrows()
        ]
    }

    # Criar Noul independente para cada tweet
    # Instrução: "É um desastre real? (não para metáforas, ficção, promoção)"
    questions = {
        f"tweet_{i}": Noul(
            instructions=(
                f"Does `tweets[{i}].text` describe or report a real-world "
                "disaster or emergency event? "
                "Return yes for actual events such as fires, floods, earthquakes, "
                "storms, accidents, explosions, attacks, evacuations, casualties "
                "or similar emergencies. "
                "Return no when disaster-related language is metaphorical, "
                "figurative, humorous, promotional, fictional, or otherwise "
                "does not describe a real-world disaster or emergency."
            )
        )
        for i in range(len(batch))
    }

    # Chamada única à API com todas as perguntas
    response = client.system_one(
        state=state,
        questions=questions,
    )

    # Extrair P(desastre) para cada tweet
    probabilities = [
        response.answers[f"tweet_{i}"].noul
        for i in range(len(batch))
    ]

    return probabilities

In [ ]:
def predict_dataset(df, batch_size=32):
    """
    Classificar dataset inteiro em batches.
    
    Args:
        df: DataFrame com tweets
        batch_size: Número de tweets por chamada API
    
    Returns:
        Array de probabilidades (0-1)
    """
    
    probabilities = []

    with TypeSafeClient() as client:

        # Iterar em batches
        for start in tqdm(range(0, len(df), batch_size)):

            end = min(
                start + batch_size,
                len(df)
            )

            batch = df.iloc[start:end]

            # Chamar Jev com batch
            batch_probabilities = predict_batch(
                client,
                batch
            )

            probabilities.extend(
                batch_probabilities
            )

            print(
                f"{end}/{len(df)} tweets classificados"
            )

    return np.array(probabilities)

In [ ]:
## Predição

# Executar predições com batch_size=32 (eficiência)
y_score = predict_dataset(
    df_eval,
    batch_size=BATCH_SIZE
)

In [ ]:
# Primeiras 10 probabilidades (deve ser ~0.5 para ambíguo, 0-0.2 para não-desastre, 0.8+ para desastre)
y_score[:10]

In [ ]:
# Converter probabilidades em predições binárias usando threshold
y_pred = (
    y_score >= THRESHOLD
).astype(int)

In [ ]:
# Labels verdadeiros da avaliação
y_true = df_eval["target"].to_numpy()

In [ ]:
# Criar DataFrame de resultados com scores e predições
results = df_eval[
    [
        "id",
        "text",
        "target"
    ]
].copy()

results["jev_probability"] = y_score
results["jev_prediction"] = y_pred
results.head(20)

In [ ]:
## Avaliação

# Calcular métricas principais
accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred
)

recall = recall_score(
    y_true,
    y_pred
)

f1 = f1_score(
    y_true,
    y_pred
)

roc_auc = roc_auc_score(
    y_true,
    y_score
)

In [ ]:
# Resumo de métricas
metrics = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC AUC"
    ],
    "Valor": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]
})

metrics

In [ ]:
# Relatório detalhado por classe
print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Não desastre",
            "Desastre"
        ]
    )
)

In [ ]:
## Visualizações

# Matriz de confusão
cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cbar=False,
    xticklabels=[
        "Não desastre",
        "Desastre"
    ],
    yticklabels=[
        "Não desastre",
        "Desastre"
    ],
)

plt.xlabel("Predito")
plt.ylabel("Real")
plt.title(
    f"Matriz de Confusão — Jev (threshold={THRESHOLD})"
)

plt.show()

In [ ]:
# Curva ROC (AUC = 0.837 indica bom discriminante)
fpr, tpr, thresholds = roc_curve(
    y_true,
    y_score
)

auc = roc_auc_score(
    y_true,
    y_score
)

plt.figure(figsize=(7, 6))

plt.plot(
    fpr,
    tpr,
    label=f"Jev — AUC = {auc:.3f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Classificador aleatório"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC — Jev")
plt.legend()

plt.show()

In [ ]:
# Curva Precision-Recall (melhor para dados desbalanceados)
precision_curve, recall_curve, thresholds_pr = (
    precision_recall_curve(
        y_true,
        y_score
    )
)

average_precision = average_precision_score(
    y_true,
    y_score
)

plt.figure(figsize=(7, 6))

plt.plot(
    recall_curve,
    precision_curve,
    label=f"AP = {average_precision:.3f}"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall — Jev")
plt.legend()

plt.show()

In [ ]:
# Distribuição de probabilidades por classe verdadeira
plt.figure(figsize=(10, 6))

sns.histplot(
    data=results,
    x="jev_probability",
    hue="target",
    bins=20,
    kde=True,
    stat="density",
    common_norm=False,
)

plt.axvline(
    THRESHOLD,
    linestyle="--"
)

plt.xlabel("Noul — P(desastre)")
plt.title(
    "Distribuição das probabilidades produzidas pelo Jev"
)

plt.show()

In [ ]:
## Análise de Erros

# False positives: predito desastre mas era não-desastre
# Exemplo: linguagem figurada que pareceu real ao Jev
false_positives = results[
    (results["target"] == 0)
    &
    (results["jev_prediction"] == 1)
].sort_values(
    "jev_probability",
    ascending=False
)

false_positives[
    [
        "text",
        "target",
        "jev_probability"
    ]
].head(20)

In [ ]:
# False negatives: predito não-desastre mas era desastre
# Exemplo: desastre real descrito de forma vaga ou literal
false_negatives = results[
    (results["target"] == 1)
    &
    (results["jev_prediction"] == 0)
].sort_values(
    "jev_probability"
)

false_negatives[
    [
        "text",
        "target",
        "jev_probability"
    ]
].head(20)

In [ ]:
## Otimização de Threshold

# Testar diferentes thresholds e calcular F1 para cada
thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

f1_scores = []

for threshold in thresholds:
    
    # Converter score em predição com threshold
    predictions = (
        y_score >= threshold
    ).astype(int)

    # Calcular F1 score
    score = f1_score(
        y_true,
        predictions
    )

    f1_scores.append(score)

In [ ]:
# Compilar resultados de threshold × F1
threshold_results = pd.DataFrame({
    "threshold": thresholds,
    "f1": f1_scores
})

In [ ]:
# Encontrar threshold com melhor F1 score
best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_row  # threshold=0.34 dá F1=0.759

In [ ]:
# Visualizar threshold × F1 (encontrar pico)
plt.figure(figsize=(9, 5))

sns.lineplot(
    data=threshold_results,
    x="threshold",
    y="f1"
)

plt.axvline(
    best_row["threshold"],
    linestyle="--"
)

plt.xlabel("Threshold")
plt.ylabel("F1")
plt.title("Threshold × F1 — Jev")

plt.show()